<img src="img/viu_logo.png" width="200">

## 01MIAR - Ficheros

![logo](img/python_logo.png)

*Ivan Fuertes / Franklin Alvarez*

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Como leer este notebook (version mejorada)</b>

Este notebook fue **enriquecido** con bloques de teoria detallada y ejercicios para que practiques **leyendo y escribiendo ficheros** desde Python.

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AZUL = TEORIA</b></div> -> explicacion en palabras simples antes de cada concepto.

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>AMARILLO = TU TURNO</b></div> -> ejercicio para resolver. Tienes una celda vacia donde escribir tu codigo.

<div style="background-color:#ebfbee;border-left:6px solid #2f9e44;padding:6px 10px;border-radius:4px;margin:4px 0;display:inline-block;"><b>VERDE = SOLUCION</b></div> -> codigo de referencia comentado. Destapalo SOLO despues de intentar.

> **Promesa:** el codigo y la teoria del profesor original NO se han tocado. Solo se han ANADIDO bloques nuevos.

</div>

<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

En esta clase aprenderas a **interactuar con ficheros del disco**. Es uno de los temas mas practicos: casi cualquier programa real necesita leer datos de entrada (CSVs, JSONs, logs) y guardar resultados.

**Temas que cubriremos:**
- `open(...)`, modos de apertura (`r`, `w`, `a`, `b`, `+`, `t`).
- `with` (context manager) y por que es la forma correcta.
- `read`, `readline`, `readlines`, iteracion linea a linea.
- `write`, `writelines`, `flush`.
- Codificacion: `encoding='utf-8'` y por que importa.
- Modulo `csv` (`reader`, `writer`, `DictReader`, `DictWriter`).
- Modulo `json` (`load`, `dump`, `loads`, `dumps`).
- Modulo `pickle` (serializar objetos Python).
- `pathlib` y `os.path` para rutas portables.

**Importante:** todos los ejercicios usaran la carpeta `res/` que ya existe en el proyecto, o crearan archivos temporales que luego eliminaremos.

</div>

# Acceso a ficheros

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Que es leer y escribir un fichero?</b>

Un **fichero** (o archivo) es una **secuencia de bytes** guardada en el disco con un nombre. Puede ser:

- **Texto** (`.txt`, `.csv`, `.json`, `.py`, `.md`...): bytes que representan caracteres legibles.
- **Binario** (`.png`, `.jpg`, `.zip`, `.pkl`, `.exe`...): bytes que NO representan texto.

#### El ciclo basico de trabajo con un fichero

```
1. ABRIR    -> open(ruta, modo)         devuelve un objeto file
2. LEER o ESCRIBIR -> .read(), .write()  hacer trabajo
3. CERRAR   -> .close()                 liberar el recurso
```

**Por que cerrar es importante?**
- Si no cierras, los datos pueden quedar **en buffer** sin escribir realmente.
- Otros programas pueden no poder abrir el fichero (queda bloqueado).
- Consumes un *file descriptor* del sistema operativo.

**La buena noticia:** el bloque `with` (que veras en seguida) cierra el fichero **automaticamente** por ti, incluso si hay errores. Es la forma recomendada en Python.

</div>

## Ficheros

* *File* es un tipo de objeto predefinido en Python (*built-in*).
* Permite acceder a ficheros desde programas Python.
* Los ficheros son de un tipo especial:
    * Son *built-in*, pero no son ni *números*, ni *secuencias*, ni *mappings*. Tampoco responden a operadores en expresiones.
* La función *open* permite crear objetos de tipo fichero.

Formato general para abrir un fichero:

```
afile = open(filename, mode)
```

* *mode* es opcional. Por defecto, los ficheros se abren en modo lectura.
* Los datos leidos de un fichero siempre se obtienen en formato *string*. Lo mismo ocurre con escritura.
* Los ficheros se deben cerrar invocando *close* (liberación de recursos).

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; <code>with</code>: el bloque que cierra el fichero por ti</b>

El **context manager** `with` es la **forma correcta y recomendada** de trabajar con ficheros en Python.

#### Comparacion lado a lado

```python
# FORMA ANTIGUA (no recomendada)
f = open('archivo.txt', encoding='utf-8')
contenido = f.read()
f.close()                # facil de olvidar
                         # Y SI HAY UNA EXCEPCION ENTRE OPEN Y CLOSE?
                         # El fichero queda abierto.

# FORMA RECOMENDADA
with open('archivo.txt', encoding='utf-8') as f:
    contenido = f.read()
# AL SALIR DEL with (incluso si hay error), Python cierra f automaticamente.
```

#### Que es exactamente un context manager?

Un objeto que sabe **`__enter__`** (al entrar al `with`) y **`__exit__`** (al salir, incluso por excepcion). Para ficheros, `__exit__` llama a `.close()`. Tu te despreocupas.

#### Multiples ficheros en el mismo `with`

Hay **dos formas**:

```python
# Forma 1: comas
with open('a.txt') as fa, open('b.txt') as fb:
    ...

# Forma 2 (Python 3.10+): parentesis
with (
    open('a.txt') as fa,
    open('b.txt') as fb,
):
    ...
```

#### Buenas practicas

- SIEMPRE usa `with` salvo que tengas una razon muy concreta para no usarlo.
- Mantenlo lo mas corto posible: abre, lee/escribe lo que necesites, sal del bloque.
- Si necesitas el contenido fuera del `with`, leelo dentro y guarda el resultado:
  ```python
  with open('a.txt', encoding='utf-8') as f:
      datos = f.read()
  # aqui ya no necesitas f, pero datos sigue disponible
  ```

</div>

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; <code>open()</code>: la firma completa</b>

La funcion incorporada `open()` tiene mas argumentos de los que parecen. Aqui los importantes:

```python
open(file, mode='r', encoding=None, newline=None, buffering=-1)
```

| Parametro | Para que sirve |
|---|---|
| `file` | Ruta al fichero (string o `pathlib.Path`). |
| `mode` | Modo de apertura: `'r'`, `'w'`, `'a'`, con `'b'` o `'t'` y `'+'`. |
| `encoding` | Codificacion de texto. **Recomendado: `'utf-8'`** (ver abajo). |
| `newline` | Como se manejan los saltos de linea. Importante en CSV. |
| `buffering` | Tamano del buffer en bytes. Por defecto, automatico. |

#### Que devuelve `open()`?

Un **objeto fichero** (`io.TextIOWrapper` para texto, `io.BufferedReader` para binario). Sobre ese objeto llamas a metodos como `.read()`, `.readline()`, `.write()`, `.close()`.

#### Por que casi siempre poner `encoding='utf-8'`?

Si no lo pones, Python usa la codificacion **por defecto del sistema**, que en **Windows suele ser `cp1252`** y en Linux/Mac `utf-8`. Esto causa errores raros con tildes, emojis o cualquier caracter no ingles.

```python
# MAL (puede fallar en Windows con tildes):
open('archivo.txt')

# BIEN (portable):
open('archivo.txt', encoding='utf-8')
```

**Regla:** SIEMPRE especifica `encoding='utf-8'` salvo que sepas que el fichero usa otra codificacion.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 1</b>

Abre el fichero `res/one_line.txt` con `open()`, lee TODO su contenido con `.read()` y muestralo. Recuerda **cerrar** el fichero con `.close()` al final.

> Debe imprimir la unica linea que contiene ese fichero.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# # Abrimos el fichero en modo lectura (por defecto 'r')
# fichero = open('res/one_line.txt', encoding='utf-8')
# # Leemos todo el contenido como un solo string
# contenido = fichero.read()
# print(contenido)
# # IMPORTANTE: cerrar para liberar el recurso
# fichero.close()


In [ ]:
# Lectura desde fichero usando método 'read'. Devuelve todo el contenido del fichero.
mi_fichero = open('./res/one_line.txt')
print(mi_fichero.read())
mi_fichero.close()

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Tres formas de leer un fichero</b>

Una vez abierto un fichero en modo texto, tienes **tres metodos** para leerlo:

| Metodo | Que devuelve | Cuando usarlo |
|---|---|---|
| `.read()` | TODO el contenido como un solo string | Ficheros pequenos. **Cuidado con ficheros grandes**: carga todo en RAM. |
| `.read(n)` | Los proximos `n` caracteres | Leer en trozos (chunks). |
| `.readline()` | UNA linea (con su `\n` final) | Procesar linea a linea controlando el flujo. |
| `.readlines()` | Lista de TODAS las lineas | Necesitas todas las lineas en memoria. |
| `for linea in f:` | Itera linea a linea, eficiente | **La forma recomendada** para ficheros de cualquier tamano. |

#### Por que iterar con `for` es lo mejor?

Porque Python lee **una linea a la vez**, sin cargar el fichero entero en RAM. Funciona igual con un fichero de 10 KB que con uno de 10 GB.

```python
with open('grande.txt', encoding='utf-8') as f:
    for linea in f:
        # procesar linea (incluye el '\n' al final)
        ...
```

#### El `\n` de mas

`.readline()` y la iteracion con `for` devuelven cada linea **con el salto de linea final**. Si haces `print(linea)`, veras una linea en blanco extra (porque `print` ya anade su propio `\n`).

**Soluciones:**
- `print(linea, end='')` -> no anadir el `\n` extra de print.
- `linea.rstrip()` o `linea.rstrip('\n')` -> quitar el `\n` de la linea.

</div>

In [ ]:
# Lectura linea a linea a través del bucle 'for'.
mi_fichero = open('res/multiple_lines.txt')
for linea in mi_fichero:
    print(f"{linea}", end='')
mi_fichero.close()

In [ ]:
# leer de un archivo a una lista.
mi_fichero = open('res/multiple_lines.txt')
lineas = mi_fichero.readlines()
mi_fichero.close()

print(lineas)

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 2</b>

Tienes el fichero `res/multiple_lines.txt`. Abre con `open(...)` (modo `r`, encoding utf-8) y resuelve **TRES tareas**:

1. Cuenta **cuantas lineas** tiene el fichero (usa `.readlines()` y `len(...)`).
2. Imprime **solo la primera linea** del fichero (usa `.readline()` despues de reabrir).
3. Imprime cada linea **sin la doble linea en blanco** (usa el truco `end=''` o `rstrip()`).

Acuerdate de cerrar (o usa `with` si ya lo conoces).

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# # 1) Contar lineas con readlines
# f = open('res/multiple_lines.txt', encoding='utf-8')
# lineas = f.readlines()       # lista de strings (cada uno termina en \n)
# print('Numero de lineas:', len(lineas))
# f.close()
#
# # 2) Solo la primera linea con readline
# f = open('res/multiple_lines.txt', encoding='utf-8')
# primera = f.readline()        # devuelve UNA linea con su \n final
# print('Primera linea:', primera, end='')
# f.close()
#
# # 3) Iterar sin doble salto de linea
# f = open('res/multiple_lines.txt', encoding='utf-8')
# for linea in f:
#     print(linea, end='')      # no anadimos otro \n
# f.close()


- Resolución de paths independiente de plataforma

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Rutas portables: <code>os.path.join</code> y <code>pathlib</code></b>

Cada sistema operativo separa las carpetas de forma distinta:

- **Windows:** `res\datos\fichero.txt` (barra invertida `\`).
- **Linux / Mac:** `res/datos/fichero.txt` (barra normal `/`).

Si escribes la ruta a mano con barras, tu codigo **no sera portable**. Python te ofrece dos formas correctas:

#### 1. La forma clasica: <code>os.path</code>

```python
import os

ruta = os.path.join('res', 'datos', 'fichero.txt')
# Windows: 'res\\datos\\fichero.txt'
# Linux:   'res/datos/fichero.txt'

os.path.exists(ruta)       # True/False
os.path.isfile(ruta)       # True si existe y es fichero
os.path.isdir(ruta)        # True si es carpeta
os.path.basename(ruta)     # 'fichero.txt'
os.path.dirname(ruta)      # 'res/datos'
os.path.splitext(ruta)     # ('res/datos/fichero', '.txt')
```

#### 2. La forma moderna: <code>pathlib</code> (Python 3.4+)

`pathlib` te da un objeto **`Path`** orientado a objetos, mucho mas comodo:

```python
from pathlib import Path

ruta = Path('res') / 'datos' / 'fichero.txt'   # uses '/' como operador!

ruta.exists()        # True/False
ruta.is_file()       # True/False
ruta.is_dir()        # True/False
ruta.name            # 'fichero.txt'
ruta.stem            # 'fichero'
ruta.suffix          # '.txt'
ruta.parent          # Path('res/datos')

# Leer / escribir directamente desde el Path:
texto = ruta.read_text(encoding='utf-8')
ruta.write_text('hola', encoding='utf-8')

# Listar contenido de una carpeta:
for f in Path('res').iterdir():
    print(f)

# Buscar con patrones:
for csv in Path('res').glob('*.csv'):
    print(csv)
```

**Recomendacion:** usa `pathlib` en codigo nuevo. Es mas legible y potente.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 3</b>

Usando **`pathlib`**:

1. Crea un `Path` que apunte a `res/multiple_lines.txt`.
2. Comprueba que existe con `.exists()`.
3. Imprime su `.name`, `.stem` y `.suffix`.
4. Lee su contenido completo con `.read_text(encoding='utf-8')` y muestra los primeros 50 caracteres.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# from pathlib import Path
#
# # 1) Construimos el path con el operador /
# ruta = Path('res') / 'multiple_lines.txt'
#
# # 2) Comprobamos existencia
# print('Existe:', ruta.exists())
#
# # 3) Partes del path
# print('Name  :', ruta.name)     # multiple_lines.txt
# print('Stem  :', ruta.stem)     # multiple_lines
# print('Suffix:', ruta.suffix)   # .txt
#
# # 4) Leemos el contenido directamente
# texto = ruta.read_text(encoding='utf-8')
# print('Primeros 50 chars:', repr(texto[:50]))


In [ ]:
import os

dataset_file_path = ["res", "multiple_lines.txt"]
ruta = os.path.join(*dataset_file_path)
print(ruta)

* *close* autómatico con sentencia *with.* Esta es la forma habitual de leer de fichero en Python, context manager

In [ ]:
with open(ruta) as mi_fichero:
    for linea in mi_fichero:
        print(linea, end= '')

* Se puede abrir varios ficheros en un mismo *with*.

In [ ]:
#abrir varios ficheros en el mismo with
ruta1 = os.path.join("res", "one_line.txt")
ruta2 = os.path.join("res", "multiple_lines.txt")
with open(ruta1) as fichero1, open(ruta2) as fichero2:
    print(fichero1.readlines())
    print(fichero2.readlines())

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 4</b>

Crea un fichero **temporal** llamado `res/o_saludo.txt` y dentro escribe la frase `Hola mundo desde Python`. Usa `with open(...)` con modo `'w'` y `encoding='utf-8'`.

Despues, en otro `with`, **lee** el fichero y muestra su contenido.

Por ultimo, eliminalo con `os.remove(ruta)` o `Path(ruta).unlink()` (lo que prefieras) para no dejar basura.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import os
# ruta = os.path.join('res', 'o_saludo.txt')
#
# # Escritura: 'w' crea o sobrescribe
# with open(ruta, 'w', encoding='utf-8') as f:
#     f.write('Hola mundo desde Python')
#
# # Lectura
# with open(ruta, 'r', encoding='utf-8') as f:
#     contenido = f.read()
# print(contenido)
#
# # Limpieza: borramos el fichero
# os.remove(ruta)
# print('Fichero eliminado.')


## Modos de acceso

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Modos de apertura: la tabla maestra</b>

Los **modos** de `open()` son letras que combinas para indicar QUE quieres hacer y COMO.

#### Letra principal (que vas a hacer)

| Modo | Significado | Si NO existe el fichero | Si SI existe |
|---|---|---|---|
| `'r'` | Leer (read) | Error `FileNotFoundError` | Lee desde el principio |
| `'w'` | Escribir (write) | Lo crea | **Lo VACIA** (cuidado!) |
| `'a'` | Anadir (append) | Lo crea | Anade al final, sin tocar el contenido previo |
| `'x'` | Crear exclusivo | Lo crea | Error `FileExistsError` (no sobrescribe) |

#### Letra secundaria (formato del contenido)

| Modo | Significado |
|---|---|
| `'t'` | Texto (DEFECTO). Lee/escribe `str` y maneja `\n`. |
| `'b'` | Binario. Lee/escribe `bytes`. Usar para imagenes, audio, pickle... |

#### El extra: <code>+</code> (lectura **Y** escritura)

| Modo | Que permite |
|---|---|
| `'r+'` | Lee y escribe. NO trunca. Falla si no existe. |
| `'w+'` | Lee y escribe. **TRUNCA** al abrir. |
| `'a+'` | Lee y anade. Crea si no existe. La escritura siempre va al final. |

#### Combinaciones tipicas

| Quiero... | Modo |
|---|---|
| Leer un .txt | `'r'` (o `'rt'`) |
| Leer una imagen | `'rb'` |
| Crear un .txt nuevo | `'w'` |
| Anadir lineas a un log | `'a'` |
| Escribir bytes (pickle) | `'wb'` |
| Leer bytes (pickle) | `'rb'` |

> **Cuidado mortal:** `'w'` **borra todo** el contenido previo del fichero al abrirlo. Si el fichero contenia datos importantes, se perdieron. Si no estas seguro, usa `'a'` o `'x'`.

</div>

<div style="background-color:#fff0f0;border-left:6px solid #e03131;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#7d1818;font-size:14px;">CUIDADO</b>

**Resumen visual de los peligros:**

- Abrir con `'w'` **siempre** vacia el fichero. No hay confirmacion.
- Abrir con `'r'` un fichero que no existe lanza `FileNotFoundError`.
- Abrir con `'x'` un fichero que YA existe lanza `FileExistsError`. Esto es bueno: te protege de sobrescribir.
- Si no especificas `encoding`, en Windows leeras tildes mal. SIEMPRE pon `encoding='utf-8'`.
- En modo binario (`'b'`) NO se puede usar `encoding`: trabajas con bytes.

</div>

* Al crear un objeto de tipo *File* se puede espeficiar el modo de acceso (lectura/escritura).

| Modo Acceso | Descripción |
|:---------|:-----|
| r | Solo Lectura |
| w | Solo Escritura (Borra si el archivo ya existe) |
| x | Solo Escritura (Falla si el archivo ya existe) |
| a | Crea Fichero (Si existe lo abre y se añade al final) |
| r+ | Lectura y Escritura |
| b | Se puede añadir a otros modos para acceso binario
| t | Acceso para archivos de texto (default) |

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 5</b>

Practica los modos. Crea (en orden):

1. Un fichero `res/o_log.txt` con modo `'w'` que contenga `linea inicial\n`.
2. Abre el MISMO fichero con modo `'a'` (append) y escribe `linea anadida 1\n` y luego `linea anadida 2\n`.
3. Abre el fichero con modo `'r'` y muestra todo su contenido.
4. Intenta abrir con modo `'x'`: deberia fallar con `FileExistsError` (capturalo con `try/except` y muestra `'Ya existe'`).
5. Borra el fichero al final.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import os
# ruta = os.path.join('res', 'o_log.txt')
#
# # 1) Crear con 'w' (sobrescribe si existe)
# with open(ruta, 'w', encoding='utf-8') as f:
#     f.write('linea inicial\n')
#
# # 2) Anadir con 'a' (NO toca lo previo)
# with open(ruta, 'a', encoding='utf-8') as f:
#     f.write('linea anadida 1\n')
#     f.write('linea anadida 2\n')
#
# # 3) Leer con 'r'
# with open(ruta, 'r', encoding='utf-8') as f:
#     print(f.read())
#
# # 4) Intentar 'x' sobre fichero existente
# try:
#     with open(ruta, 'x', encoding='utf-8') as f:
#         f.write('no llegara')
# except FileExistsError:
#     print('Ya existe')
#
# # 5) Limpieza
# os.remove(ruta)


## Acceso para escritura

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Escribir en ficheros: <code>write</code> y <code>writelines</code></b>

Para escribir en un fichero abierto en modo `'w'`, `'a'` o `'x'`:

#### `.write(string)`

Escribe **un solo string**. Devuelve el numero de caracteres escritos.

```python
with open('out.txt', 'w', encoding='utf-8') as f:
    f.write('Hola\n')
    f.write('Mundo\n')
```

> `.write()` **NO anade** salto de linea automatico. Si quieres una linea por escritura, debes incluir tu `\n`.

#### `.writelines(iterable_de_strings)`

Escribe **una secuencia de strings** uno detras de otro. **TAMPOCO anade `\n`** entre ellos: tu debes ponerlos.

```python
lineas = ['uno\n', 'dos\n', 'tres\n']
with open('out.txt', 'w', encoding='utf-8') as f:
    f.writelines(lineas)
```

> Pese a su nombre, `writelines` no anade saltos. Si tus strings no terminan en `\n`, todo se escribe pegado en una sola linea gigante.

#### Truco util: usar `print` con un fichero

`print` tiene un argumento `file` que redirige la salida a un fichero (anadiendo el salto de linea por ti):

```python
with open('out.txt', 'w', encoding='utf-8') as f:
    print('Hola', file=f)
    print('Mundo', file=f)
    print('A=', 5, 'B=', 10, file=f)   # 'A= 5 B= 10'
```

#### Que pasa si escribes en `'r'`?

Error: `io.UnsupportedOperation: not writable`. El fichero esta abierto solo para leer.

#### Anadir vs sobrescribir

| Quiero... | Modo |
|---|---|
| Empezar con un fichero vacio (borrar lo viejo) | `'w'` |
| Conservar lo viejo y anadir al final | `'a'` |
| Que falle si el fichero ya existe | `'x'` |

</div>

In [ ]:
def crear_lista(tamanyo):
    lista = []
    for i in range(tamanyo):
        lista.append(str(i) + '\n')
    return lista

crear_lista(10)

In [ ]:
ruta = os.path.join("res", "o_dummy.txt")
with open(ruta, 'at') as fichero:
    fichero.write('Cabecera\n')
    lista = crear_lista(20)
    fichero.writelines(lista)

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 6</b>

Crea un fichero `res/o_tabla.txt` con la **tabla de multiplicar del 7** (del 7x1 al 7x10), una linea por cada producto, con este formato:

```
7 x 1 = 7
7 x 2 = 14
...
7 x 10 = 70
```

Despues lee el fichero y muestralo. Borra el fichero al final.

> Pista: usa un bucle `for i in range(1, 11):` y un f-string.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import os
# ruta = os.path.join('res', 'o_tabla.txt')
#
# # Escritura
# with open(ruta, 'w', encoding='utf-8') as f:
#     for i in range(1, 11):
#         # f-string con el resultado y un salto de linea
#         f.write(f'7 x {i} = {7 * i}\n')
#
# # Lectura
# with open(ruta, 'r', encoding='utf-8') as f:
#     print(f.read())
#
# # Limpieza
# os.remove(ruta)


#### Buffering

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Buffering: por que tus datos a veces no aparecen</b>

Cuando llamas a `f.write('hola')`, **Python NO escribe inmediatamente en el disco**. Guarda los datos en un **buffer** (una zona de memoria) y los escribe cuando:

1. El buffer se llena (por defecto, unos KB).
2. Tu llamas a `f.flush()` (forzar escritura).
3. Tu cierras el fichero con `f.close()` o sales del `with`.

#### Por que existe el buffer?

Porque escribir en disco es **mucho mas lento** que escribir en memoria. Si cada `write()` golpeara el disco, el rendimiento seria pesimo. El buffer agrupa varias escrituras y las hace todas de golpe.

#### Cuando me afecta?

```python
f = open('out.txt', 'w', encoding='utf-8')
f.write('hola')
# AQUI: si abres out.txt desde otro programa, esta VACIO.
# Los datos siguen en el buffer.
f.close()
# AHORA: ya esta en disco.
```

#### Como forzar la escritura?

```python
f.flush()       # vacia el buffer al disco SIN cerrar el fichero
f.close()       # vacia el buffer y cierra
```

Si usas `with`, `flush` ocurre automaticamente al salir del bloque.

#### Casos en los que importa

- **Logs en tiempo real:** quieres ver las lineas en cuanto se escriben. Usa `flush=True` en `print()`:
  ```python
  print('avance...', file=f, flush=True)
  ```
- **Procesos largos:** si tu programa se cuelga sin cerrar el fichero, perderas los datos del buffer.
- **Otro programa lee el fichero mientras tu escribes:** usa `flush()` para que vea los cambios.

</div>

* Por defecto, el texto que transfieres desde tu programa a un fichero no se guarda en disco inmediatamente. Se almacena en un buffer.
* Acciones como cerrar un fichero o invocar el método **flush** fuerzan que se transfiera el contenido del buffer a disco.

In [ ]:
ruta = os.path.join("res", "FicheroParaEscritura.txt")
fichero_write = open(ruta, 'w')
fichero_write.write('foo')

In [ ]:
fichero_read = open(ruta, "r")
print(fichero_read.readlines())

In [ ]:
fichero_write.flush()

print(fichero_read.readlines())

fichero_write.close()
fichero_read.close()

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 7</b>

Reproduce el efecto del buffer **a mano**:

1. Abre `res/o_buffer.txt` con `'w'`, escribe `'pendiente'` y NO cierres aun.
2. Abre el mismo fichero con `'r'` y muestra su contenido (lo veras vacio o sin la palabra).
3. Llama a `flush()` sobre el fichero de escritura.
4. Vuelve a leer y muestra el contenido (ahora SI veras `'pendiente'`).
5. Cierra ambos ficheros y borra el archivo.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import os
# ruta = os.path.join('res', 'o_buffer.txt')
#
# # 1) Abrimos para escribir y escribimos sin cerrar
# fw = open(ruta, 'w', encoding='utf-8')
# fw.write('pendiente')
#
# # 2) Abrimos otro handle para leer (los datos siguen en buffer)
# fr = open(ruta, 'r', encoding='utf-8')
# print('Antes de flush:', repr(fr.read()))   # vacio
#
# # 3) Forzamos la escritura al disco
# fw.flush()
#
# # 4) Releemos (necesitamos volver al inicio con seek o reabrir)
# fr.seek(0)
# print('Despues de flush:', repr(fr.read()))  # 'pendiente'
#
# # 5) Cerrar y limpiar
# fw.close()
# fr.close()
# os.remove(ruta)


## Archivos CSV

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; CSV: un formato simple para datos tabulares</b>

**CSV** = *Comma-Separated Values*. Es **texto plano** donde cada linea es una fila y los campos van separados por comas (o por `;`, tabuladores...).

#### Ejemplo

```
nombre,edad,ciudad
Ana,30,Madrid
Pedro,25,Barcelona
Lucia,35,Valencia
```

#### Por que NO leer un CSV con `read()` y `split(',')`?

Porque parece simple pero tiene casos peligrosos:
- Un campo puede llevar comas dentro: `"apellido, nombre"` (entre comillas).
- Un campo puede llevar saltos de linea.
- Algunos CSVs usan `;` o `\t` como separador.
- Los strings se pueden escapar con `""`.

**El modulo `csv` de la libreria estandar** maneja todo eso por ti.

#### Las cuatro herramientas del modulo

| Clase | Para que |
|---|---|
| `csv.reader(f)` | Lee filas como **listas** `['Ana', '30', 'Madrid']` |
| `csv.writer(f)` | Escribe filas pasandole listas |
| `csv.DictReader(f)` | Lee filas como **diccionarios** `{'nombre':'Ana', 'edad':'30', ...}` |
| `csv.DictWriter(f, fieldnames=...)` | Escribe filas pasandole diccionarios |

#### Patron tipico de lectura

```python
import csv

with open('datos.csv', 'r', encoding='utf-8', newline='') as f:
    lector = csv.reader(f)
    cabecera = next(lector)        # primera fila = nombres de columna
    for fila in lector:
        print(fila)                # ['Ana', '30', 'Madrid']
```

#### Patron tipico con DictReader

```python
import csv

with open('datos.csv', 'r', encoding='utf-8', newline='') as f:
    lector = csv.DictReader(f)
    for fila in lector:
        # fila ya es un dict con las claves de la cabecera
        print(fila['nombre'], fila['edad'])
```

#### Patron tipico de escritura

```python
import csv

filas = [
    ['Ana', 30, 'Madrid'],
    ['Pedro', 25, 'Barcelona'],
]

with open('out.csv', 'w', encoding='utf-8', newline='') as f:
    escritor = csv.writer(f)
    escritor.writerow(['nombre', 'edad', 'ciudad'])  # cabecera
    escritor.writerows(filas)                         # todas las filas
```

#### Por que <code>newline=''</code>?

Si NO lo pones, en Windows aparecen **lineas en blanco extra** entre filas. El modulo `csv` ya gestiona los saltos de linea, asi que le decimos a `open` que no transforme nada.

> **Regla de oro:** abre ficheros CSV SIEMPRE con `newline=''` y `encoding='utf-8'`.

</div>

* Python permite leer datos de ficheros CSV y también escribir ficheros en este formato.
* Popular formato en ciencia de datos.

In [ ]:
# tabla_operaciones.csv contiene valores separados por comas
import csv
ruta = os.path.join("res", "tabla_operaciones.csv")

with open(ruta) as fichero:
    data_reader = csv.reader(fichero, delimiter=',')
    for linea in data_reader:
        print(linea[0] + '  ----  ' + linea[1])

In [ ]:
ruta = os.path.join("res", "tabla_operaciones.csv")
ruta_o = os.path.join("res", "o_tabla_operaciones.csv")

with open(ruta) as f1, open(ruta_o, 'w') as f2:
    data_reader = csv.reader(f1, delimiter = ',')
    csv_writer = csv.writer(f2, delimiter = '|')
    for line in data_reader:
        # print(line)
        csv_writer.writerow(line)

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; JSON: el formato estrella para datos estructurados</b>

**JSON** = *JavaScript Object Notation*. Es un formato de texto que se ha convertido en el estandar para intercambiar datos entre sistemas (APIs web, configuracion, almacenamiento ligero).

#### Por que es popular?

- Es **legible por humanos**.
- Mapea **directamente** a tipos basicos de Python (dict, list, str, int, float, bool, None).
- Lo entienden practicamente todos los lenguajes de programacion.

#### Equivalencias JSON <-> Python

| JSON | Python |
|---|---|
| `{"a": 1}` | `dict` |
| `[1, 2, 3]` | `list` |
| `"hola"` | `str` |
| `42` / `3.14` | `int` / `float` |
| `true` / `false` | `True` / `False` |
| `null` | `None` |

> **Cuidado:** JSON usa `true`/`false`/`null` en minuscula. Python usa `True`/`False`/`None`. El modulo `json` traduce por ti.

#### Las cuatro funciones del modulo `json`

| Funcion | Que hace | Trabaja con |
|---|---|---|
| `json.dumps(obj)` | Convierte un objeto Python a string JSON | string |
| `json.loads(string)` | Convierte un string JSON a objeto Python | string |
| `json.dump(obj, f)` | Convierte y **escribe** en un fichero abierto | fichero |
| `json.load(f)` | **Lee** un fichero y convierte a objeto Python | fichero |

> Nemotecnia: las que terminan en `s` trabajan con **s**trings. Las otras con ficheros.

#### Ejemplos

```python
import json

datos = {
    'nombre': 'Ana',
    'edad': 30,
    'aficiones': ['leer', 'correr'],
    'casado': False,
    'hijos': None,
}

# 1) A string
texto = json.dumps(datos, ensure_ascii=False, indent=2)
print(texto)

# 2) De string a objeto
obj = json.loads('{"a": 1, "b": [10, 20]}')
print(obj['b'][1])   # 20

# 3) Escribir a fichero
with open('datos.json', 'w', encoding='utf-8') as f:
    json.dump(datos, f, ensure_ascii=False, indent=2)

# 4) Leer desde fichero
with open('datos.json', 'r', encoding='utf-8') as f:
    leido = json.load(f)
print(leido['nombre'])
```

#### Argumentos utiles

| Argumento | Para que |
|---|---|
| `indent=2` | Formatea el JSON con sangrias bonitas (legible). Sin esto, todo en una linea. |
| `ensure_ascii=False` | Permite caracteres no ASCII (utiles para utf-8). Por defecto, escapa todo a `\u00...`. |
| `sort_keys=True` | Ordena las claves alfabeticamente. |
| `default=str` | Convierte tipos no serializables (ej: `datetime`) llamando a `str(obj)`. |

#### Limitaciones de JSON

JSON **NO sabe** serializar:
- Tuplas (las convierte a listas).
- Sets, datetime, objetos personalizados.
- Funciones, clases.

Para esas cosas necesitas `pickle` o convertir manualmente.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 9</b>

Crea un diccionario `usuario` con estas claves: `nombre`, `edad`, `intereses` (lista de strings), `activo` (bool), `apellido_segundo` (None).

1. Escribelo a `res/o_usuario.json` con **`json.dump`**, `indent=2` y `ensure_ascii=False`.
2. Vuelve a leerlo con **`json.load`** y comprueba que `type(leido)` es `dict`.
3. Imprime el JSON resultante usando **`json.dumps(leido, indent=2)`** para ver el formato bonito.
4. Borra el fichero.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import json, os
# ruta = os.path.join('res', 'o_usuario.json')
#
# usuario = {
#     'nombre': 'Ana',
#     'edad': 30,
#     'intereses': ['leer', 'correr', 'python'],
#     'activo': True,
#     'apellido_segundo': None,
# }
#
# # 1) Escribir a fichero
# with open(ruta, 'w', encoding='utf-8') as f:
#     json.dump(usuario, f, indent=2, ensure_ascii=False)
#
# # 2) Leer de fichero
# with open(ruta, 'r', encoding='utf-8') as f:
#     leido = json.load(f)
# print('Tipo leido:', type(leido))     # <class 'dict'>
#
# # 3) Bonito
# print(json.dumps(leido, indent=2, ensure_ascii=False))
#
# # 4) Limpieza
# os.remove(ruta)


<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Pickle: serializar CUALQUIER objeto Python</b>

**Pickle** es la solucion de Python para guardar y cargar **objetos arbitrarios** (incluyendo cosas que JSON no puede manejar como tuplas, sets, objetos de clases personalizadas, modelos de machine learning entrenados, etc.).

#### JSON vs Pickle

| Aspecto | JSON | Pickle |
|---|---|---|
| Formato | Texto legible | Binario (no humano) |
| Portable a otros lenguajes | Si | **NO** (solo Python) |
| Tipos soportados | Basicos | **Casi todo** |
| Seguridad | Seguro | **PELIGROSO** (codigo arbitrario) |
| Velocidad | Mas lento | Rapido |
| Uso tipico | APIs, config | Cache, modelos ML |

#### CUIDADO con la seguridad

**NUNCA cargues un fichero pickle de origen desconocido.** Pickle puede ejecutar codigo Python arbitrario al deserializar. Si alguien malicioso te pasa un .pkl, puede tomar control de tu maquina.

#### Las cuatro funciones del modulo `pickle`

| Funcion | Que hace |
|---|---|
| `pickle.dumps(obj)` | Serializa a bytes (no a string) |
| `pickle.loads(b)` | Deserializa de bytes |
| `pickle.dump(obj, f)` | Serializa y escribe a fichero (modo `'wb'`) |
| `pickle.load(f)` | Lee y deserializa de fichero (modo `'rb'`) |

> Pickle trabaja con **BYTES**, no con texto. Por eso los modos son `'wb'` y `'rb'`. NO uses `encoding`.

#### Ejemplos

```python
import pickle

datos = {
    'tupla': (1, 2, 3),       # JSON la convertiria en lista
    'set': {10, 20, 30},      # JSON no puede manejar sets
    'lista_anidada': [[1, 2], [3, 4]],
}

# 1) Escribir a fichero (modo binario!)
with open('datos.pkl', 'wb') as f:
    pickle.dump(datos, f)

# 2) Leer de fichero
with open('datos.pkl', 'rb') as f:
    leido = pickle.load(f)

print(type(leido['tupla']))   # <class 'tuple'>  (mantiene el tipo)
print(type(leido['set']))     # <class 'set'>
```

#### Cuando usar pickle?

- **Cache de calculos largos:** computar algo lento una vez, guardarlo, recuperarlo en proximas ejecuciones.
- **Modelos de Machine Learning:** despues de entrenar `model.fit(...)`, guardar con `pickle.dump(model, f)` y cargarlo en produccion.
- **Estado intermedio de un programa.**

#### Cuando NO usar pickle?

- Para compartir datos con otros lenguajes (usa JSON o CSV).
- Para configuracion editable a mano (usa JSON o YAML).
- Para datos de fuente no confiable (peligro de codigo malicioso).

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 10</b>

Crea un objeto Python con tipos que JSON NO sabria manejar:

```python
mi_objeto = {
    'tupla': (1, 2, 3),
    'conjunto': {10, 20, 30},
    'matriz': [[1, 2], [3, 4]],
}
```

1. Guardalo en `res/o_objeto.pkl` usando **`pickle.dump`** (modo `'wb'`).
2. Vuelve a cargarlo con **`pickle.load`** (modo `'rb'`).
3. Comprueba que `type(leido['tupla']) is tuple` y `type(leido['conjunto']) is set` (los tipos se mantienen, a diferencia de JSON).
4. Borra el fichero al final.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import pickle, os
# ruta = os.path.join('res', 'o_objeto.pkl')
#
# mi_objeto = {
#     'tupla': (1, 2, 3),
#     'conjunto': {10, 20, 30},
#     'matriz': [[1, 2], [3, 4]],
# }
#
# # 1) Escribimos a fichero EN MODO BINARIO ('wb')
# with open(ruta, 'wb') as f:
#     pickle.dump(mi_objeto, f)
#
# # 2) Leemos de fichero EN MODO BINARIO ('rb')
# with open(ruta, 'rb') as f:
#     leido = pickle.load(f)
#
# # 3) Comprobamos que los tipos se han conservado
# print(type(leido['tupla']) is tuple)       # True
# print(type(leido['conjunto']) is set)      # True
# print(leido)
#
# # 4) Limpieza
# os.remove(ruta)


<div style="background-color:#f3f0ff;border-left:6px solid #7048e8;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#4c2d9e;font-size:14px;">PARA RECORDAR</b>

**Resumen visual: que formato elegir?**

| Tienes... | Quieres... | Usa |
|---|---|---|
| Datos tabulares (filas y columnas) | Compartir con Excel / R / otros | **CSV** |
| Diccionario / lista de tipos basicos | API web / configuracion legible | **JSON** |
| Objeto Python complejo (tuplas, sets, modelos ML, clases) | Solo Python -> Python | **Pickle** |
| Texto libre (logs, notas) | Lectura humana | **TXT** plano |
| Imagen / audio / pdf | Almacenar tal cual | **modo binario `'rb'`/`'wb'`** |

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 8</b>

Tienes datos de productos. Realiza lo siguiente:

1. Crea una **lista de diccionarios** con 3 productos: cada dict tiene `id`, `nombre`, `precio`.
2. Escribelos en `res/o_productos.csv` usando **`csv.DictWriter`**. Recuerda escribir la cabecera con `.writeheader()`.
3. Vuelve a abrirlo con **`csv.DictReader`** y muestra cada producto en una linea como `{nombre} -> {precio} EUR`.
4. Borra el fichero al final.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import csv, os
# ruta = os.path.join('res', 'o_productos.csv')
#
# productos = [
#     {'id': 1, 'nombre': 'pan',    'precio': 1.20},
#     {'id': 2, 'nombre': 'leche',  'precio': 0.95},
#     {'id': 3, 'nombre': 'huevos', 'precio': 2.50},
# ]
#
# # 1+2) Escritura con DictWriter
# with open(ruta, 'w', encoding='utf-8', newline='') as f:
#     campos = ['id', 'nombre', 'precio']
#     escritor = csv.DictWriter(f, fieldnames=campos)
#     escritor.writeheader()
#     escritor.writerows(productos)
#
# # 3) Lectura con DictReader
# with open(ruta, 'r', encoding='utf-8', newline='') as f:
#     lector = csv.DictReader(f)
#     for fila in lector:
#         # ojo: los valores leidos son strings
#         print(f"{fila['nombre']} -> {fila['precio']} EUR")
#
# # 4) Limpieza
# os.remove(ruta)


# Docstrings

* Python permite adjuntar documentación a los objetos e inspeccionarla a través de la línea de comandos o durante la ejecución del programa.
* Los docstrings se almacenan en el atributo *\_\_doc\_\_* de cada objeto.
* El valor de dicho atributo se puede mostrar por medio de la función *help*.

In [ ]:
help(open)

* No es necesario editar este atributo directamente.
* Para asociar un docstring a un objeto basta con escribir el texto (entre triples comillas) al principio de los modulos, funciones o clases, antes del codigo ejecutable.

In [ ]:
def funcion_de_prueba():
    """Esta es la documentación de la función de prueba"""
    pass

help(funcion_de_prueba)

* Accediendo al atributo *\_\_doc\_\_* sólo se obtiene el docstring.

In [ ]:
print(funcion_de_prueba.__doc__)

Como se puede observar, *help* añade información adicional. Por ejemplo, para entidades más grandes, *help* muestra el docstring dividido en secciones.

In [ ]:
import sys
help(sys)

Los docstrings que ocupan más de una línea suelen tener estas partes:

* Resumen en la primera línea.
* Línea en blanco.
* Descripción detallada.
* Otra línea en blanco antes del código.

In [ ]:
def funcion_de_prueba2():
    """Esta es la línea para el resumen.
    
    Este es el párrafo donde se puede escribir una descripción más
    detallada de la función. Observa que el resumen y la descripción
    detallada van separados por una línea en blanco. También hay
    otra línea en blanco antes de empezar el código.
    """
    
    pass

help(funcion_de_prueba2)

Algunos comentarios adicionales:

* Comentarios con '#' suelen asociarse a expresiones sencillas, instrucciones individuales o pequeños bloques de código.
* Los docstrings son más apropiados para construcciones de más alto nivel: funciones, clases y módulos.

## Sphinx

* Herramienta de documentación.
* Especialmente útil para sistemas grandes.
* Da soporte a una gran variedad de formatos de salida: HTML, LaTeX, ePub, ...

Web de [Sphinx](https://www.sphinx-doc.org/).

[Proyectos](https://www.sphinx-doc.org/en/master/examples.html) que usan Sphinx.

## Estilo

* Es importante que los docstrings sean consistentes.
* El equipo de desarrollo debe acordar un formato y seguirlo rigurosamente.

#### Google Python Style Guide

Google define la siguiente [guía de estilo](https://github.com/google/styleguide/blob/gh-pages/pyguide.md#38-comments-and-docstrings).

Contiene reglas para los docstrings de:

* Módulos.
* Métodos y funciones.
* Clases.

## Ejercicios

1. Escribe una función que reciba una ruta de un fichero de texto y una cadena de caracteres a buscar y determine si la cadena aparece en el fichero.

2. Escribe una función que reciba una lista, una ruta destino y un número *n*. La función debe crear un fichero en la ruta especificada. El contenido del fichero serán los primeros *n* elementos de la lista. La función debe controlar de manera apropiada los posibles valores de *n* que estén fuera de rango.

3. Escribe una función que reciba una ruta de un fichero de texto devuelva un diccionario con la frecuencia de aparición de cada palabra. Ejemplo: un fichero que contenga la frase 'es mejor que venga que que no venga' devolverá el siguiente diccionario: {'es' : 1, 'mejor' : 1, 'que' : 3, 'venga' : 2, 'no' : 1}. Para dividir un string en palabras puedes hacer uso del método *split*.

In [ ]:
# Ejercicio 1

def existe_en_fichero(ruta_fichero, patron_a_buscar):
    with open(ruta_fichero) as fichero:
        for linea in fichero:
            if patron_a_buscar in linea:
                return True
    return False
        
        
print(existe_en_fichero('res/VariasLineas.txt', 'tres')) # True
print(existe_en_fichero('res/VariasLineas.txt', 'hola')) # False

In [ ]:
# Ejercicio 2

def escribir_objetos(lista, ruta_fichero, n):
    with open(ruta_fichero, 'w') as fichero:
        try:
            if(n < 0):
                raise IndexError
                
            for i in range(n):
                fichero.write(f"{lista[i]}\n")
            
            print(f"Fichero creado satisfactoriamente. Se escribieron {n} elementos en el fichero.")
        except IndexError:
            print(f"Acceso fuera de rango. Valor n incorrecto: {n}.")
            
escribir_objetos(['a', 'b', 'c', 'd', 'e', 'f'], 'res/Ejercicio2.txt', 2)

In [ ]:
# Ejercicio 3

def anyadir_palabra(frecuencias, palabra):
    if palabra in frecuencias:
        frecuencias[palabra] += 1
    else:
        frecuencias[palabra] = 1

def obtener_frecuencias(ruta_fichero):
    frecuencias = {}
    with open(ruta_fichero) as fichero:
        for linea in fichero:
            for palabra in linea.split():
                anyadir_palabra(frecuencias, palabra)
    return frecuencias

obtener_frecuencias('res/Ejercicio3.txt')

<div style="background-color:#e7f5ff;border-left:6px solid #1c7ed6;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#1864ab;font-size:14px;">TEORIA &middot; Resumen final y patron canonico</b>

Si te quedas con UNA sola idea de esta clase, que sea esta:

```python
from pathlib import Path
import json   # o csv, o pickle, segun el caso

ruta = Path('res') / 'datos.json'

# LECTURA
with open(ruta, 'r', encoding='utf-8') as f:
    datos = json.load(f)

# ... trabajar con datos ...

# ESCRITURA
ruta_salida = Path('res') / 'salida.json'
with open(ruta_salida, 'w', encoding='utf-8') as f:
    json.dump(datos, f, indent=2, ensure_ascii=False)
```

**Checklist al manipular ficheros:**

- [ ] Uso `with` (cierra automaticamente).
- [ ] Especifico `encoding='utf-8'` para texto.
- [ ] Para CSV anado `newline=''`.
- [ ] Para binarios uso `'rb'` / `'wb'` (sin encoding).
- [ ] No uso `'w'` accidentalmente cuando queria `'a'` (sobrescribir vs anadir).
- [ ] Manejo errores comunes con `try/except`: `FileNotFoundError`, `PermissionError`, `IsADirectoryError`.
- [ ] Para datos tabulares uso `csv`. Para datos estructurados, `json`. Para objetos Python, `pickle`.

</div>

<div style="background-color:#fff9db;border-left:6px solid #f59f00;padding:12px 16px;border-radius:6px;margin:6px 0;"><b style="color:#a05a00;font-size:14px;">TU TURNO 11</b>

**Ejercicio integrador.** Tienes que construir una mini-pipeline:

1. Define una **lista de diccionarios** con 3 alumnos: cada uno con `id`, `nombre`, `notas` (lista de 3 enteros).
2. Calcula la nota **media** de cada alumno y anadela a su dict como clave `media`.
3. Guarda la lista resultante en `res/o_alumnos.json` (con `indent=2`).
4. Tambien escribe una version CSV en `res/o_alumnos.csv` con columnas `id,nombre,media` (sin la lista `notas`). Usa `csv.DictWriter`.
5. Vuelve a leer el JSON y muestra los alumnos cuya media sea **>= 7**.
6. Borra los dos ficheros.

<i>Escribe tu codigo en la celda que sigue.</i></div>

In [ ]:
# Escribe aqui tu codigo:



In [ ]:
# === Solucion sugerida ===
# import json, csv, os
#
# # 1) Datos de partida
# alumnos = [
#     {'id': 1, 'nombre': 'Ana',   'notas': [8, 9, 7]},
#     {'id': 2, 'nombre': 'Pedro', 'notas': [5, 6, 4]},
#     {'id': 3, 'nombre': 'Lucia', 'notas': [9, 9, 10]},
# ]
#
# # 2) Calculamos la media
# for a in alumnos:
#     a['media'] = sum(a['notas']) / len(a['notas'])
#
# # 3) Guardar a JSON
# ruta_json = os.path.join('res', 'o_alumnos.json')
# with open(ruta_json, 'w', encoding='utf-8') as f:
#     json.dump(alumnos, f, indent=2, ensure_ascii=False)
#
# # 4) Guardar a CSV (sin la lista 'notas')
# ruta_csv = os.path.join('res', 'o_alumnos.csv')
# with open(ruta_csv, 'w', encoding='utf-8', newline='') as f:
#     escritor = csv.DictWriter(f, fieldnames=['id', 'nombre', 'media'])
#     escritor.writeheader()
#     for a in alumnos:
#         # Solo las columnas que queremos
#         escritor.writerow({'id': a['id'], 'nombre': a['nombre'], 'media': a['media']})
#
# # 5) Releemos JSON y filtramos
# with open(ruta_json, 'r', encoding='utf-8') as f:
#     leido = json.load(f)
# aprobados = [a for a in leido if a['media'] >= 7]
# print('Alumnos con media >= 7:')
# for a in aprobados:
#     print(f"  {a['nombre']}: {a['media']:.2f}")
#
# # 6) Limpieza
# os.remove(ruta_json)
# os.remove(ruta_csv)
